In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("silver-layer").getOrCreate()

spark

In [2]:
BRONZE_PATH = "/home/jovyan/work/submission/output/bronze"
SILVER_PATH = "/home/jovyan/work/submission/output/silver"

In [3]:
orders_df = spark.read.csv(
    f"{BRONZE_PATH}/orders.csv",
    header=True,
    inferSchema=True
)

orders_df.show(5)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+----------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|        _ingested_at|_source_endpoint|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+--------------------+----------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|2026-03-27 16:03:...|          orders|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|201

In [4]:
from pyspark.sql.functions import to_timestamp

orders_df = orders_df.withColumn(
    "order_purchase_timestamp",
    to_timestamp("order_purchase_timestamp")
).withColumn(
    "order_approved_at",
    to_timestamp("order_approved_at")
).withColumn(
    "order_delivered_carrier_date",
    to_timestamp("order_delivered_carrier_date")
).withColumn(
    "order_delivered_customer_date",
    to_timestamp("order_delivered_customer_date")
).withColumn(
    "order_estimated_delivery_date",
    to_timestamp("order_estimated_delivery_date")
)

orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_endpoint: string (nullable = true)



In [5]:
orders_df = orders_df.dropDuplicates(["order_id"])

print("After dedup:", orders_df.count())

After dedup: 99441


In [6]:
from pyspark.sql.functions import col

orders_df = orders_df.withColumn(
    "_is_valid",
    col("order_delivered_customer_date") >= col("order_purchase_timestamp")
)

orders_df.select("order_id", "_is_valid").show(5)

+--------------------+---------+
|            order_id|_is_valid|
+--------------------+---------+
|f373335aac9a659de...|     NULL|
|118045506e1c1dda0...|     true|
|cc66dee6fbc18bb79...|     true|
|f44cb69655f8e4d13...|     true|
|edcc6b79e8394346b...|     true|
+--------------------+---------+
only showing top 5 rows



In [7]:
orders_df.write.mode("overwrite").parquet(f"{SILVER_PATH}/orders")

In [8]:
order_items_df = spark.read.csv(
    f"{BRONZE_PATH}/order_items.csv",
    header=True,
    inferSchema=True
)

order_items_df = order_items_df.dropDuplicates()

order_items_df = order_items_df.withColumn(
    "_is_valid",
    order_items_df.price > 0
)

order_items_df.write.mode("overwrite").parquet(f"{SILVER_PATH}/order_items")

print("order_items silver saved")

order_items silver saved


In [9]:
customers_df = spark.read.csv(
    f"{BRONZE_PATH}/customers.csv",
    header=True,
    inferSchema=True
)

customers_df = customers_df.dropDuplicates(["customer_id"])

customers_df = customers_df.withColumn(
    "_is_valid",
    customers_df.customer_id.isNotNull()
)

customers_df.write.mode("overwrite").parquet(f"{SILVER_PATH}/customers")

print("customers silver saved")

customers silver saved


In [10]:
products_df = spark.read.csv(
    f"{BRONZE_PATH}/products.csv",
    header=True,
    inferSchema=True
)

products_df = products_df.dropDuplicates(["product_id"])

products_df = products_df.withColumn(
    "_is_valid",
    products_df.product_id.isNotNull()
)

products_df.write.mode("overwrite").parquet(f"{SILVER_PATH}/products")

print("products silver saved")

products silver saved


In [11]:
sellers_df = spark.read.csv(
    f"{BRONZE_PATH}/sellers.csv",
    header=True,
    inferSchema=True
)

sellers_df = sellers_df.dropDuplicates(["seller_id"])

sellers_df = sellers_df.withColumn(
    "_is_valid",
    sellers_df.seller_id.isNotNull()
)

sellers_df.write.mode("overwrite").parquet(f"{SILVER_PATH}/sellers")

print("sellers silver saved")

sellers silver saved


In [12]:
payments_df = spark.read.csv(
    f"{BRONZE_PATH}/payments.csv",
    header=True,
    inferSchema=True
)

payments_df = payments_df.dropDuplicates()

payments_df = payments_df.withColumn(
    "_is_valid",
    payments_df.payment_value > 0
)

payments_df.write.mode("overwrite").parquet(f"{SILVER_PATH}/payments")

print("payments silver saved")

payments silver saved
